In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

input_dir = Path("/kaggle/input/datasets/emmagasparato/eurocrops")
output_dir = Path("/kaggle/working/eurocrops_csv")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for file in input_dir.rglob("*.parquet"):
    df = gpd.read_parquet(file)

    # Convert the geometrr in WKT text
    if isinstance(df, gpd.GeoDataFrame):
        df = df.to_crs("EPSG:4326")
        df["geometry"] = df.geometry.to_wkt()
        df = pd.DataFrame(df)

    output_file = output_dir / f"{file.stem}.csv"
    df.to_csv(output_file, index=False)

    print(f"Converted: {file.name} -> {output_file.name}")

In [ ]:
crop_rotations_2020 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2020.csv")
crop_rotations_2021 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2021.csv")
crop_rotations_2022 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2022.csv")
crop_rotations_2023 = pd.read_csv("/kaggle/working/eurocrops_csv/iti1_2023.csv")

In [ ]:
mapping = pd.read_csv("/kaggle/input/datasets/emmagasparato/eurocrops/eurocrops.csv")

print(mapping.columns.tolist())
display(mapping.head())

mapping = mapping[mapping["nuts"].str.lower() == "iti1"].copy()
print("Crops in the mapping:", len(mapping))

crop_rotations_2020["original_code"] = crop_rotations_2020["original_code"].astype(str)
mapping["original_code"] = mapping["original_code"].astype(str)

crop_rotations_2020 = crop_rotations_2020.merge(
    mapping[["original_code", "original_name", "translated_name", "hcat4_name"]].drop_duplicates(),
    on="original_code",
    how="left",
    validate="many_to_one"
)

display(crop_rotations_2020.head())

In [ ]:
print("Crops without correspondence:",
      crop_rotations_2020["hcat4_name"].isna().sum())

display(
    crop_rotations_2020[
        ["cropfield", "original_code", "translated_name", "hcat4_name", "area_ha"]
    ].head(10)
)

In [ ]:
crop_areas = (
    crop_rotations_2020.groupby("hcat4_name", dropna=False)["area_ha"]
    .sum()
    .sort_values(ascending=False)
)

display(crop_areas.head(20))

In [ ]:
mapping["original_code"] = mapping["original_code"].astype(str)

crop_rotations = {2020: crop_rotations_2020}

for year in [2021, 2022, 2023]:
    df = pd.read_csv(f"/kaggle/working/eurocrops_csv/iti1_{year}.csv")
    df["original_code"] = df["original_code"].astype(str)

    df = df.merge(
        mapping[["original_code", "original_name", "translated_name", "hcat4_name"]].drop_duplicates(),
        on="original_code",
        how="left",
        validate="many_to_one"
    )

    crop_rotations[year] = df
    print(f"{year}: {len(df)} lots, {df['hcat4_name'].isna().sum()} without mapping")

for year in [2021, 2022, 2023]:
    crop_rotations[year].to_csv(
        f"/kaggle/working/iti1_{year}_mapped.csv", index=False
    )

In [ ]:
crop_rotations_all = pd.concat([
    crop_rotations_2020.assign(year=2020),
    crop_rotations_2021.assign(year=2021),
    crop_rotations_2022.assign(year=2022),
    crop_rotations_2023.assign(year=2023)
], ignore_index=True)

display(crop_rotations_all.head())
print("Dimensions:", crop_rotations_all.shape)

In [ ]:
crop_rotations_all.to_csv(
    "/kaggle/working/eurocrops_toscana_2020_2023.csv",
    index=False
)
print(crop_rotations_all.groupby("year").size())
print(crop_rotations_all.isna().sum())

In [ ]:
crop_rotations_all.head()

In [ ]:
import geopandas as gpd

base = Path("/kaggle/input/datasets/emmagasparato/eurocrops")

mapping = pd.read_csv(base / "eurocrops.csv", dtype={"original_code": str})
mapping = mapping[mapping["nuts"].str.lower() == "iti1"]
mapping = mapping[["original_code", "hcat4_name"]].drop_duplicates("original_code")

fields = {}

for year in [2020, 2021, 2022, 2023]:
    gdf = gpd.read_parquet(base / f"iti1_{year}.parquet")
    gdf["original_code"] = gdf["original_code"].astype(str)
    gdf = gdf.merge(mapping, on="original_code", how="left")
    gdf = gdf.to_crs("EPSG:3035").reset_index(drop=True)
    gdf["field_id"] = gdf.index

    fields[year] = gdf
    print(year, len(gdf))

In [ ]:
def match_fields(left, right, threshold=0.8):
    a = left[["field_id", "geometry"]].copy()
    b = right[["field_id", "geometry"]].copy()

    pairs = gpd.sjoin(a, b, how="inner", predicate="intersects",
                      lsuffix="left", rsuffix="right")

    if pairs.empty:
        return pd.DataFrame(columns=["id_left", "id_right", "overlap"])

    ids_a = pairs["field_id_left"].to_numpy()
    ids_b = pairs["field_id_right"].to_numpy()

    ga = a.geometry.iloc[ids_a].reset_index(drop=True)
    gb = b.geometry.iloc[ids_b].reset_index(drop=True)

    intersection = ga.intersection(gb).area.to_numpy()
    overlap = intersection / pd.concat([
        ga.area, gb.area
    ], axis=1).max(axis=1).to_numpy()

    matches = pd.DataFrame({
        "id_left": ids_a,
        "id_right": ids_b,
        "overlap": overlap
    })

    matches = matches[matches["overlap"] >= threshold]
    matches = matches.sort_values("overlap", ascending=False)
    matches = matches.drop_duplicates("id_left").drop_duplicates("id_right")

    return matches.reset_index(drop=True)

In [ ]:
matches = {}

for year in [2020, 2021, 2022]:
    m = match_fields(fields[year], fields[year + 1])
    matches[year] = m
    print(f"{year} → {year+1}: {len(m)} associated fields")

In [ ]:
sequences = matches[2020][["id_left", "id_right"]].rename(
    columns={"id_left": "id_2020", "id_right": "id_2021"}
)

for year in [2021, 2022]:
    m = matches[year][["id_left", "id_right"]].rename(
        columns={"id_left": f"id_{year}", "id_right": f"id_{year+1}"}
    )
    sequences = sequences.merge(m, on=f"id_{year}", how="inner")

for year in [2020, 2021, 2022, 2023]:
    crops = fields[year][["field_id", "hcat4_name"]].rename(
        columns={"field_id": f"id_{year}", "hcat4_name": f"crop_{year}"}
    )
    sequences = sequences.merge(crops, on=f"id_{year}", how="left")

sequences["field_id"] = range(1, len(sequences) + 1)

print("Complete sequences:", len(sequences))
display(sequences.head())

In [ ]:
sequences.to_csv("/kaggle/working/crop_rotations_2020_2023.csv", index=False)

from IPython.display import FileLink
display(FileLink("/kaggle/working/crop_rotations_2020_2023.csv"))

In [ ]:
selected_crops = [
    'Alfalfa', 'Avocado', 'Avocado subtropical ecotype', 'Buckwheat',
    'Barley', 'Biomass sorghum', 'Cabbage', 'Camelina', 'Carrot',
    'Chickpea', 'Citrus', 'Cotton', 'Carinata', 'Finger millet',
    'Flax fibre', 'Foxtail millet', 'Groundnut', 'Miscanthus',
    'Maize', 'Silage maize', 'Oat', 'Olive', 'Onion', 'Dry pea',
    'Phaseolus bean', 'Pasture grasses', 'Reed canary grass',
    'Rapeseed', 'Rye', 'Sunflower', 'Solaris energy tobacco',
    'Soybean', 'Sorghum', 'Sugar beet', 'Switchgrass', 'Tef',
    'Tobacco', 'Tomato', 'Triticale', 'Wheat', 'Watermelon',
    'White potato'
]
print(sorted(crop_rotations_all["hcat4_name"].dropna().unique()))

In [ ]:
from difflib import get_close_matches

available = crop_rotations_all["hcat4_name"].dropna().unique().tolist()

for crop in selected_crops:
    matches = get_close_matches(
        crop.lower().replace(" ", "_"),
        available,
        n=3,
        cutoff=0.4
    )
    print(f"{crop}: {matches}")

In [ ]:
crop_mapping = {
    "maize_corn_popcorn": "Maize",
    "olive": "Olive",
    "sunflower": "Sunflower"
}

filtered_crops = crop_rotations_all[
    crop_rotations_all["hcat4_name"].isin(crop_mapping)
].copy()

filtered_crops["crop_name"] = filtered_crops["hcat4_name"].map(crop_mapping)

print("Original observations:", len(crop_rotations_all))
print("Filtered observations:", len(filtered_crops))

display(filtered_crops.head())

In [ ]:
columns_to_drop = ["original_name", "translated_name", "hcat4_name"]

filtered_crops = filtered_crops.drop(columns=columns_to_drop, errors="ignore")

filtered_crops.to_csv(
    "/kaggle/working/eurocrops_selected_crops.csv",
    index=False
)

display(filtered_crops.head())

**geometry** - coordinates which describe the shape and the position of the cropfield

**cropfield** - id of the cropfield in the dataset 

**off_id** - original administrative id

**area_ha** - surface of the cropfield in hectares

**off_area** - original reported surface